# Last inn data

In [ ]:
import pandas as pd

df = pd.read_json("lånekassen_data.json")
df["number_of_sents"] = df.fulltext.apply(lambda x: len([e for e in x if len(e)>2]))
assert len(set(df.doc_hash)) == len(df)

nynorske = df[df.lang == "nno"].copy()
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"].copy()
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

In [ ]:
from collections import defaultdict

nynorske_sentences = defaultdict(list)
for t, df_ in nynorske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        nynorske_sentences["text"].append(t)
        nynorske_sentences["doc_hashes"].append(set(df_.doc_hash))
        nynorske_sentences["urls"].append(set(df_.url))

nynorske_flat = pd.DataFrame(nynorske_sentences)

bokmålske_sentences = defaultdict(list)
for t, df_ in bokmålske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        bokmålske_sentences["text"].append(t)
        bokmålske_sentences["doc_hashes"].append(set(df_.doc_hash))
        bokmålske_sentences["urls"].append(set(df_.url))

bokmålske_flat = pd.DataFrame(bokmålske_sentences)

In [ ]:
over_en_nn = nynorske_flat[nynorske_flat.doc_hashes.apply(len) > 1]
over_en_bm = bokmålske_flat[bokmålske_flat.doc_hashes.apply(len) > 1]

print(f"""
Det er {len(nynorske_flat)} unike nynorske setninger/linjer 
Det er {len(over_en_nn)} nynorske setninger som forekommer i mer enn ett dokument ({round((len(over_en_nn) / len(nynorske_flat))* 100, 2)}%)

Det er {len(bokmålske_flat)} unike setninger/linjer på bokmål
Det er {len(over_en_bm)} setninger på bokmål som forekommer i mer enn ett dokument ({round((len(over_en_bm) / len(bokmålske_flat))* 100, 2)}%)
""")

In [ ]:
len(set(nynorske_flat.doc_hashes.apply(tuple))), len(nynorske)

# Bygg graf basert på setnigns-alignment

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import numpy as np

base_path = "/nb_sbert/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    model = SentenceTransformer('NbAiLab/nb-sbert-base', device="cuda")
    basemodel_max_len = model[0].auto_model.config.max_position_embeddings
    model.max_seq_length = basemodel_max_len
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)


search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=3)

## Vektet graf med score

In [ ]:
import networkx as nx

G = nx.MultiDiGraph()

for e in nynorske_flat.itertuples():
    i = e.Index
    seen_doc_hashes = set()
    for res in search_result[i]:
        bm_doc_hashes = bokmålske_flat.iloc[res["corpus_id"]].doc_hashes
        score = res["score"]

        for nn_h in e.doc_hashes:
            for bm_h in bm_doc_hashes:
                if bm_h in seen_doc_hashes:
                    # only add one edge between two documents for each sentence
                    continue
                seen_doc_hashes.add(bm_h)
                G.add_edge(nn_h, bm_h, weight=score, no_bm_docs=len(bm_doc_hashes))

In [ ]:
from collections import defaultdict

doc_matches = defaultdict(list)
threshold = 0.75

for e in nynorske.itertuples():
    node = e.doc_hash
    weights_per_node = defaultdict(list)
    for nn_node, bm_node, weight in G.edges(node, data="weight"):
        weights_per_node[bm_node].append(weight)
    for bm_node, scores in weights_per_node.items():
        weighted_mean_score = np.mean(scores) * (len(scores)/e.number_of_sents)
        if weighted_mean_score > threshold:
            doc_matches["nynorsk_doc_hash"].append(node)
            doc_matches["bokmål_doc_hash"].append(bm_node)
            doc_matches["weighted_mean_score"].append(weighted_mean_score)

matches = pd.DataFrame(doc_matches)
matches = matches.merge(nynorske[["doc_hash", "fulltext", "url"]], left_on="nynorsk_doc_hash", right_on="doc_hash").merge(bokmålske[["doc_hash", "fulltext", "url"]], left_on="bokmål_doc_hash", right_on="doc_hash", suffixes=["_nn", "_bm"])

matches

In [ ]:
for e in matches.itertuples():
    if e.weighted_mean_score < 0.8:
        print(e.weighted_mean_score)
        print(e.fulltext_nn[:5])
        print("")
        print(e.fulltext_bm[:5])
        print("\n\n")

# Sammenlikn med fasit

In [ ]:
fasit = pd.read_csv("lanekassen_fasit.csv")
fasit[:10]

In [ ]:
treff = fasit[["nynorsk_doc_hash", "bokmål_doc_hash"]].merge(matches, on="nynorsk_doc_hash")
treff = treff[treff.bokmål_doc_hash_x == treff.bokmål_doc_hash_y][["weighted_mean_score", "doc_hash_nn", "fulltext_nn", "url_nn", "doc_hash_bm","fulltext_bm", "url_bm"]]

In [ ]:
len(treff), len(fasit)

### Se på hvilke dokumenter fra fasiten som ikke ble truffet

In [ ]:
traff_ikke = set(fasit.nynorsk_doc_hash) - set(treff.doc_hash_nn)
len(traff_ikke)

In [ ]:
nn_dhs = fasit[fasit.nynorsk_doc_hash.isin(traff_ikke)].nynorsk_doc_hash
bm_dhs = fasit[fasit.nynorsk_doc_hash.isin(traff_ikke)].bokmål_doc_hash

for nn_dh, bm_dh in zip(nn_dhs, bm_dhs):
    nn_txt = df[df.doc_hash == nn_dh].fulltext.item()
    bm_txt = df[df.doc_hash == bm_dh].fulltext.item()
    if len(nn_txt) == len(bm_txt):
        print(nn_txt)
        print(bm_txt)
    else:
        print(len(nn_txt), len(bm_txt))
    print("\n")


In [ ]:
len(doc_matches["nynorsk_doc_hash"])